# 🎵 Aiscern — Audio Deepfake Detector
### Kaggle P100 | wav2vec2-base + LoRA | Full 162k samples × 20 epochs

**Platform**: Kaggle (30h free/week, no session limit)  
**GPU**: P100 16GB  
**Expected time**: ~14.3h  
**Output model**: `saghi776/aiscern-audio-detector`  
**Expected accuracy**: 91% → **97%**

### Datasets used
| Dataset | Size | Type |
|---|---|---|
| ASVspoof2019-LA | 25,000 | Anti-spoofing benchmark |
| WaveFake | ~118,000 | 7 GAN vocoders on LJSpeech |
| in-the-wild deepfakes | ~20,000 | Real-world fake audio |
| Mozilla CommonVoice | 100,000 | Real human speech |
| LibriSpeech clean | 100,000 | Real human speech |


In [ ]:
# Install all dependencies
!pip install -q transformers==4.40.0 datasets peft==0.10.0 accelerate              evaluate scikit-learn soundfile librosa huggingface_hub audioread

import os, warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
print("✅ Dependencies installed")


In [ ]:
# ── CONFIG — edit HF_TOKEN then Run All ─────────────────────────
HF_TOKEN        = 'YOUR_HF_TOKEN_HERE'   # ← paste your HF token here

BASE_MODEL      = 'facebook/wav2vec2-base'
PUSH_REPO       = 'saghi776/aiscern-audio-detector'
CHECKPOINT_DIR  = './audio-checkpoints'

# Training hyperparameters (optimised for Kaggle P100 16GB)
SAMPLE_RATE     = 16000
MAX_DURATION_S  = 5          # clip to 5s — 80k samples at 16kHz
SAMPLES_PER_CLASS = 81474    # all available data (162,948 ÷ 2)
BATCH_SIZE      = 16
EPOCHS          = 20         # 20 epochs uses ~28h of Kaggle's 30h budget
LR              = 2e-4
WARMUP_RATIO    = 0.06
WEIGHT_DECAY    = 0.01
SEED            = 42

import os, torch
os.environ['HF_TOKEN'] = HF_TOKEN
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print("⚠️  No GPU detected — switch to GPU P100 in Kaggle settings")
else:
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu}  ({mem:.1f} GB VRAM)")
    print(f"   Training {SAMPLES_PER_CLASS*2:,} samples × {EPOCHS} epochs")
    print(f"   Estimated time: ~14.3h")


In [ ]:
# ── LOAD + COMBINE DATASETS ─────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Audio
import numpy as np

def normalise_label(val):
    """Map any label format to 0=real, 1=fake"""
    s = str(val).lower().strip()
    if s in ('spoof','spoofed','fake','ai','1','generated'): return 1
    if s in ('bonafide','genuine','real','human','0'): return 0
    return -1

print("Loading datasets...")
all_splits = []

# 1. ASVspoof2019 LA — gold standard benchmark (25k samples)
try:
    ds = load_dataset(
        'DynamicSuperb/AudioDeepfakeDetection_ASVspoof2019LA',
        split='train', token=HF_TOKEN, trust_remote_code=True
    )
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
    lbl_col = next((c for c in ds.column_names if 'label' in c.lower()), None)
    if lbl_col:
        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])}, 
                    remove_columns=[c for c in ds.column_names if c not in ('audio','label')])
        ds = ds.filter(lambda x: x['label'] != -1)
        all_splits.append(ds)
        print(f"  ✅ ASVspoof2019: {len(ds):,} (real={ds.filter(lambda x:x['label']==0).num_rows:,} fake={ds.filter(lambda x:x['label']==1).num_rows:,})")
except Exception as e:
    print(f"  ⚠️  ASVspoof2019 skipped: {e}")

# 2. WaveFake — 7 GAN vocoders (~118k FAKE samples)
try:
    ds = load_dataset('balt0/WaveFake', split='train', token=HF_TOKEN)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
    ds = ds.map(lambda x: {'label': 1},  # WaveFake = all fake
                remove_columns=[c for c in ds.column_names if c not in ('audio','label')])
    all_splits.append(ds)
    print(f"  ✅ WaveFake: {len(ds):,} (all fake)")
except Exception as e:
    print(f"  ⚠️  WaveFake skipped: {e}")

# 3. In-the-wild deepfakes (~20k samples)
try:
    ds = load_dataset('motheecreator/in-the-wild-audio-deepfake', 
                      split='train', token=HF_TOKEN)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
    lbl_col = next((c for c in ds.column_names if 'label' in c.lower()), None)
    if lbl_col:
        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])},
                    remove_columns=[c for c in ds.column_names if c not in ('audio','label')])
        ds = ds.filter(lambda x: x['label'] != -1)
        all_splits.append(ds)
        print(f"  ✅ In-the-wild: {len(ds):,}")
except Exception as e:
    print(f"  ⚠️  In-the-wild skipped: {e}")

# 4. Mozilla CommonVoice (REAL speech)
try:
    ds = load_dataset('mozilla-foundation/common_voice_11_0', 'en',
                      split='train', token=HF_TOKEN, trust_remote_code=True)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
    ds = ds.map(lambda x: {'label': 0},
                remove_columns=[c for c in ds.column_names if c not in ('audio','label')])
    ds = ds.shuffle(SEED).select(range(min(100000, len(ds))))
    all_splits.append(ds)
    print(f"  ✅ CommonVoice: {len(ds):,} (all real)")
except Exception as e:
    print(f"  ⚠️  CommonVoice skipped: {e}")

# 5. LibriSpeech (REAL speech)  
try:
    ds = load_dataset('openslr/librispeech_asr', 'clean',
                      split='train.100', token=HF_TOKEN, trust_remote_code=True)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
    ds = ds.map(lambda x: {'label': 0},
                remove_columns=[c for c in ds.column_names if c not in ('audio','label')])
    all_splits.append(ds)
    print(f"  ✅ LibriSpeech: {len(ds):,} (all real)")
except Exception as e:
    print(f"  ⚠️  LibriSpeech skipped: {e}")

# ── Combine + balance ────────────────────────────────────────────
if not all_splits:
    raise RuntimeError("No datasets loaded — check HF_TOKEN and internet access")

combined = concatenate_datasets(all_splits)
print(f"\nCombined: {len(combined):,} total")

real = combined.filter(lambda x: x['label'] == 0)
fake = combined.filter(lambda x: x['label'] == 1)
n = min(len(real), len(fake), SAMPLES_PER_CLASS)
print(f"Balancing: {n:,} per class ({n*2:,} total)")

real = real.shuffle(SEED).select(range(n))
fake = fake.shuffle(SEED).select(range(n))
balanced = concatenate_datasets([real, fake]).shuffle(SEED)

split = balanced.train_test_split(test_size=0.1, seed=SEED)
train_ds, eval_ds = split['train'], split['test']
print(f"\n✅ Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}")


In [ ]:
# ── FEATURE EXTRACTION ──────────────────────────────────────────
from transformers import Wav2Vec2FeatureExtractor
import numpy as np

extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    BASE_MODEL, token=HF_TOKEN
)
MAX_LEN = SAMPLE_RATE * MAX_DURATION_S   # 80,000

def preprocess(batch):
    arrays = []
    for item in batch['audio']:
        arr = np.array(item['array'], dtype=np.float32)
        arr = arr[:MAX_LEN]                        # truncate
        if len(arr) < SAMPLE_RATE:                 # skip very short clips
            arr = np.pad(arr, (0, SAMPLE_RATE - len(arr)))
        # Normalise amplitude
        rms = np.sqrt(np.mean(arr**2))
        if rms > 0:
            arr = arr / (rms + 1e-8) * 0.1
        arrays.append(arr)
    
    inputs = extractor(
        arrays, sampling_rate=SAMPLE_RATE,
        return_tensors='np', padding='max_length',
        max_length=MAX_LEN, truncation=True,
        return_attention_mask=True,
    )
    batch['input_values']   = inputs.input_values
    batch['attention_mask'] = inputs.attention_mask
    return batch

print("Extracting features — this takes ~15min...")
train_ds = train_ds.map(preprocess, batched=True, batch_size=32,
                         remove_columns=['audio'],
                         desc='Train features')
eval_ds  = eval_ds.map(preprocess,  batched=True, batch_size=32,
                         remove_columns=['audio'],
                         desc='Eval features')
train_ds.set_format('torch')
eval_ds.set_format('torch')
print(f"✅ Features ready: {train_ds.num_rows:,} train, {eval_ds.num_rows:,} eval")


In [ ]:
# ── MODEL + LoRA ─────────────────────────────────────────────────
from transformers import Wav2Vec2ForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'real', 1: 'fake'},
    label2id={'real': 0, 'fake': 1},
    ignore_mismatched_sizes=True,
    token=HF_TOKEN,
)

# Freeze CNN feature extractor — only fine-tune transformer attention
model.freeze_feature_extractor()

# LoRA: trains only ~500k of 95M params (0.5%)
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
model = model.to(device)
print(f"✅ Model on {device}")


In [ ]:
# ── TRAIN ────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate, numpy as np

acc_metric = evaluate.load('accuracy')
f1_metric  = evaluate.load('f1')

def compute_metrics(ep):
    preds  = np.argmax(ep.predictions, axis=-1)
    labels = ep.label_ids
    return {
        'accuracy': acc_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1':       f1_metric.compute(predictions=preds, references=labels, average='binary')['f1'],
        'fake_recall': f1_metric.compute(predictions=preds, references=labels, average=None)['f1'][1],
    }

training_args = TrainingArguments(
    output_dir              = CHECKPOINT_DIR,
    num_train_epochs        = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate           = LR,
    warmup_ratio            = WARMUP_RATIO,
    weight_decay            = WEIGHT_DECAY,
    eval_strategy           = 'epoch',
    save_strategy           = 'epoch',
    save_total_limit        = 3,
    load_best_model_at_end  = True,
    metric_for_best_model   = 'f1',
    greater_is_better       = True,
    push_to_hub             = True,
    hub_model_id            = PUSH_REPO,
    hub_token               = HF_TOKEN,
    hub_strategy            = 'every_save',
    fp16                    = (device == 'cuda'),
    dataloader_num_workers  = 4,
    report_to               = 'none',
    logging_steps           = 100,
    logging_first_step      = True,
    seed                    = SEED,
    resume_from_checkpoint  = True,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Starting training: {len(train_ds):,} samples × {EPOCHS} epochs")
print(f"Estimated time:    ~14.3h on Kaggle P100")
print(f"Checkpoints saved every epoch → hub_strategy=every_save")
print()
trainer.train()
print("\n✅ Training complete!")


In [ ]:
# ── EVALUATE + PUSH TO HUGGINGFACE ─────────────────────────────
results = trainer.evaluate()

print("\n" + "═"*50)
print("FINAL RESULTS")
print("═"*50)
print(f"  Accuracy:    {results['eval_accuracy']:.4f}  ({results['eval_accuracy']*100:.2f}%)")
print(f"  F1 Score:    {results['eval_f1']:.4f}")
print(f"  Fake Recall: {results.get('eval_fake_recall', 0):.4f}  (catches {results.get('eval_fake_recall',0)*100:.1f}% of fakes)")
print("═"*50)

# Push best model + extractor to HuggingFace
trainer.push_to_hub(commit_message=f"Audio deepfake detector — accuracy={results['eval_accuracy']:.4f}")
extractor.push_to_hub(PUSH_REPO, token=HF_TOKEN)

from huggingface_hub import HfApi
api = HfApi()

# Create model card
card = f"""---
license: apache-2.0
tags:
- audio-classification
- deepfake-detection
- wav2vec2
- lora
datasets:
- DynamicSuperb/AudioDeepfakeDetection_ASVspoof2019LA
- balt0/WaveFake
metrics:
- accuracy
- f1
model-index:
- name: aiscern-audio-detector
  results:
  - task:
      type: audio-classification
    metrics:
    - type: accuracy
      value: {results['eval_accuracy']:.4f}
    - type: f1
      value: {results['eval_f1']:.4f}
---

# Aiscern Audio Deepfake Detector

Fine-tuned `facebook/wav2vec2-base` with LoRA for audio deepfake detection.

**Accuracy**: {results['eval_accuracy']*100:.2f}%  
**F1**: {results['eval_f1']:.4f}  
**Training data**: {len(train_ds):,} samples (ASVspoof2019 + WaveFake + in-the-wild + real speech)

## Usage
```python
from transformers import pipeline
detector = pipeline('audio-classification', model='{PUSH_REPO}')
result = detector('audio.wav')
# [{{'label': 'fake', 'score': 0.97}}]
```
"""

api.upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo='README.md',
    repo_id=PUSH_REPO,
    token=HF_TOKEN,
)

print(f"\n✅ Model live at: https://huggingface.co/{PUSH_REPO}")
print(f"   Use in Aiscern: model='{PUSH_REPO}'")
